# 测试输出

## 生成代码

In [1]:
import csv
import gc
import re
import shutil
import time
import unicodedata
from pathlib import Path

import jiwer
import librosa
import pandas as pd
import torch
from IPython.display import display
from peft import PeftModel
from transformers import (
    AutoModelForSpeechSeq2Seq,
    AutoProcessor,
    BitsAndBytesConfig,
)

# ============================================================
# 1. 选择语言
# ============================================================

LANG = "fr"  # fr / de / nn

LANG_SPECS = {
    "fr": {
        "cv_root": Path(
            "/home/sigan/datasets/common_voice_27_fr"
            "/cv-corpus-27.0-2026-09-11/fr"
        ),
        "language": "french",
        "abbr": "fr",
    },
    "de": {
        "cv_root": Path(
            "/home/sigan/datasets/common_voice_27_ge"
            "cv-corpus-27.0-2026-09-11/de"
        ),
        "language": "german",
        "abbr": "de",
    },
    "nn": {
        "cv_root": Path(
            "/home/sigan/datasets/common_voice_27_nork"
            "/cv-corpus-27.0-2026-09-11/nn-NO"
        ),
        "language": "nynorsk",
        "abbr": "nn",
    },
}

CFG = LANG_SPECS[LANG]
CV_ROOT = CFG["cv_root"]
WHISPER_LANGUAGE = CFG["language"]

NUM_SAMPLES = 20
SAMPLING_RATE = 16000
MAX_NEW_TOKENS = 128
DEVICE = "cuda"

TEST_DIR = Path(
    f"/sigan/ai-ml/datasets/"
    f"common-voice-27-{LANG}-test20"
)
TEST_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. 复制 20 条测试音频
# ============================================================

with (CV_ROOT / "test.tsv").open(
    encoding="utf-8",
    newline="",
) as handle:
    reader = csv.DictReader(handle, delimiter="\t")

    test_rows = []
    selected_speakers = set()

    for row in reader:
        # 尽量选择不同说话人
        if row["client_id"] in selected_speakers:
            continue

        test_rows.append(row)
        selected_speakers.add(row["client_id"])

        if len(test_rows) == NUM_SAMPLES:
            break

for row in test_rows:
    shutil.copy2(
        CV_ROOT / "clips" / row["path"],
        TEST_DIR / row["path"],
    )

manifest_df = pd.DataFrame([
    {
        "序号": index,
        "文件名": row["path"],
        "标准答案": row["sentence"],
    }
    for index, row in enumerate(test_rows, start=1)
])

print("音频目录：", TEST_DIR)
display(manifest_df)


# ============================================================
# 3. 文本归一化和读取音频
# ============================================================

def normalize_text(text):
    text = unicodedata.normalize("NFKC", text).lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    return re.sub(r"\s+", " ", text).strip()


filenames = [row["path"] for row in test_rows]

references = [
    normalize_text(row["sentence"])
    for row in test_rows
]

audio_arrays = [
    librosa.load(
        str(TEST_DIR / row["path"]),
        sr=SAMPLING_RATE,
    )[0]
    for row in test_rows
]


# ============================================================
# 4. 运行一个模型
# ============================================================

def run_model(model_id, adapter_dir):
    processor = AutoProcessor.from_pretrained(
        model_id,
        language=WHISPER_LANGUAGE,
        task="transcribe",
    )

    forced_decoder_ids = processor.get_decoder_prompt_ids(
        language=WHISPER_LANGUAGE,
        task="transcribe",
    )

    base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id,
        quantization_config=BitsAndBytesConfig(
            load_in_8bit=True
        ),
        device_map="auto",
    )
    base_model.config.use_cache = True

    model = base_model

    if adapter_dir:
        model = PeftModel.from_pretrained(
            base_model,
            adapter_dir,
        )

    model.eval()
    model.generation_config.forced_decoder_ids = (
        forced_decoder_ids
    )

    predictions = []
    start_time = time.perf_counter()

    for index, audio in enumerate(audio_arrays, start=1):
        inputs = processor(
            audio,
            sampling_rate=SAMPLING_RATE,
            return_tensors="pt",
        )
        inputs["input_features"] = (
            inputs["input_features"]
            .to(DEVICE)
            .half()
        )

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                num_beams=1,
            )

        text = processor.batch_decode(
            output_ids,
            skip_special_tokens=True,
        )[0]

        predictions.append(normalize_text(text))
        print(f"{model_id}：{index}/{NUM_SAMPLES}")

    elapsed = time.perf_counter() - start_time

    del model
    del base_model
    del processor
    gc.collect()
    torch.cuda.empty_cache()

    return predictions, elapsed


# ============================================================
# 5. 依次评估四个候选模型
# ============================================================

MODEL_SPECS = [
    (
        "V2基座",
        "openai/whisper-large-v2",
        None,
    ),
    (
        "V2微调",
        "openai/whisper-large-v2",
        f"/sigan/ai-ml/models/"
        f"whisper-large-v2-{LANG}-lora",
    ),
    (
        "V3基座",
        "openai/whisper-large-v3",
        None,
    ),
    (
        "V3微调",
        "openai/whisper-large-v3",
        f"/sigan/ai-ml/models/"
        f"whisper-large-v3-{LANG}-lora",
    ),
]

all_predictions = {}
all_times = {}

for name, model_id, adapter_dir in MODEL_SPECS:
    print("\n开始测试：", name)

    predictions, elapsed = run_model(
        model_id,
        adapter_dir,
    )

    all_predictions[name] = predictions
    all_times[name] = elapsed


# ============================================================
# 6. 汇总 WER 和逐条结果
# ============================================================

print("\n总体结果：")

for name, predictions in all_predictions.items():
    overall_wer = jiwer.wer(
        reference=references,
        hypothesis=predictions,
    )

    print(
        f"{name}: WER={overall_wer:.4f}, "
        f"耗时={all_times[name]:.1f}秒"
    )

rows = []

for index in range(NUM_SAMPLES):
    row = {
        "序号": index + 1,
        "文件名": filenames[index],
        "标准答案": references[index],
    }

    for name, predictions in all_predictions.items():
        row[f"{name}输出"] = predictions[index]
        row[f"{name} WER"] = round(
            jiwer.wer(
                reference=references[index],
                hypothesis=predictions[index],
            ),
            4,
        )

    rows.append(row)

results_df = pd.DataFrame(rows)
display(results_df)

音频目录： /sigan/ai-ml/datasets/common-voice-27-fr-test20


,序号,文件名,标准答案
0,1,common_voice_fr_19737052.mp3,Le club italien remporte le trophée pour la tr...
1,2,common_voice_fr_20283872.mp3,Le lac est en danger d'eutrophisation à cause ...
2,3,common_voice_fr_34682414.mp3,"Sa couleur blanche est due à la bétuline, son ..."
3,4,common_voice_fr_35170021.mp3,Jusqu'à quel point est-on dans la même Amérique ?
4,5,common_voice_fr_19676492.mp3,Cette espèce se rencontre au Pérou et en Équat...
5,6,common_voice_fr_37666418.mp3,Cet épisode est le premier de la série à être ...
6,7,common_voice_fr_20468203.mp3,Elle est située en pleine campagne à un kilomè...
7,8,common_voice_fr_19677731.mp3,Le mouvement s'étend à d'autres ateliers dans ...
8,9,common_voice_fr_18747257.mp3,La parole est à Monsieur Didier Martin.
9,10,common_voice_fr_19607910.mp3,Son nom fait référence à Siméon de Jérusalem.


/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename



开始测试： V2基座


/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


openai/whisper-large-v2：1/20
openai/whisper-large-v2：2/20
openai/whisper-large-v2：3/20
openai/whisper-large-v2：4/20
openai/whisper-large-v2：5/20
openai/whisper-large-v2：6/20
openai/whisper-large-v2：7/20
openai/whisper-large-v2：8/20
openai/whisper-large-v2：9/20
openai/whisper-large-v2：10/20
openai/whisper-large-v2：11/20
openai/whisper-large-v2：12/20
openai/whisper-large-v2：13/20
openai/whisper-large-v2：14/20
openai/whisper-large-v2：15/20
openai/whisper-large-v2：16/20
openai/whisper-large-v2：17/20
openai/whisper-large-v2：18/20
openai/whisper-large-v2：19/20
openai/whisper-large-v2：20/20

开始测试： V2微调


... (3 outputs left out)

openai/whisper-large-v3：1/20
openai/whisper-large-v3：2/20
openai/whisper-large-v3：3/20
openai/whisper-large-v3：4/20
openai/whisper-large-v3：5/20
openai/whisper-large-v3：6/20
openai/whisper-large-v3：7/20
openai/whisper-large-v3：8/20
openai/whisper-large-v3：9/20
openai/whisper-large-v3：10/20
openai/whisper-large-v3：11/20
openai/whisper-large-v3：12/20
openai/whisper-large-v3：13/20
openai/whisper-large-v3：14/20
openai/whisper-large-v3：15/20
openai/whisper-large-v3：16/20
openai/whisper-large-v3：17/20
openai/whisper-large-v3：18/20
openai/whisper-large-v3：19/20
openai/whisper-large-v3：20/20

开始测试： V3微调


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


openai/whisper-large-v3：1/20
openai/whisper-large-v3：2/20
openai/whisper-large-v3：3/20
openai/whisper-large-v3：4/20
openai/whisper-large-v3：5/20
openai/whisper-large-v3：6/20
openai/whisper-large-v3：7/20
openai/whisper-large-v3：8/20
openai/whisper-large-v3：9/20
openai/whisper-large-v3：10/20
openai/whisper-large-v3：11/20
openai/whisper-large-v3：12/20
openai/whisper-large-v3：13/20
openai/whisper-large-v3：14/20
openai/whisper-large-v3：15/20
openai/whisper-large-v3：16/20
openai/whisper-large-v3：17/20
openai/whisper-large-v3：18/20
openai/whisper-large-v3：19/20
openai/whisper-large-v3：20/20

总体结果：
V2基座: WER=0.9120, 耗时=34.4秒
V2微调: WER=0.0694, 耗时=46.5秒
V3基座: WER=0.0694, 耗时=40.5秒
V3微调: WER=0.0741, 耗时=46.9秒


,序号,文件名,标准答案,V2基座输出,V2基座 WER,V2微调输出,V2微调 WER,V3基座输出,V3基座 WER,V3微调输出,V3微调 WER
0,1,common_voice_fr_19737052.mp3,le club italien remporte le trophée pour la tr...,the italian club wins the trophy for the third...,1.0000,le club italien remporte le trophée pour la tr...,0.0000,le club italien remporte le trophée pour la tr...,0.0000,le club italien remporte le trophée pour la tr...,0.0000
1,2,common_voice_fr_20283872.mp3,le lac est en danger d eutrophisation à cause ...,the lake is in danger of trophying due to huma...,0.9167,le lac est en danger d eutrophisation à cause ...,0.0000,le lac est en danger d eutrophisation à cause ...,0.0000,le lac est en danger d eutrophisation à cause ...,0.0000
2,3,common_voice_fr_34682414.mp3,sa couleur blanche est due à la bétuline son p...,its white color is due to the betuline its mai...,0.9091,sa couleur blanche est due à la bétuline son p...,0.0000,sa couleur blanche est due à la béthuline son ...,0.0909,sa couleur blanche est due à la béthuline son ...,0.0909
3,4,common_voice_fr_35170021.mp3,jusqu à quel point est on dans la même amérique,how far are we in the same america,1.0000,jusqu à quel point est on dans la même amérique,0.0000,jusqu à quel point est on dans la même amérique,0.0000,jusquà quel point est on dans la même amérique,0.2000
4,5,common_voice_fr_19676492.mp3,cette espèce se rencontre au pérou et en équateur,this species is found in peru and ecuador,1.0000,cette espèce se rencontre au pérou et en équateur,0.0000,cette espèce se rencontre au pérou et en équateur,0.0000,cette espèce se rencontre au pérou et en équateur,0.0000
5,6,common_voice_fr_37666418.mp3,cet épisode est le premier de la série à être ...,this episode is the first of the series to be ...,1.0000,cet épisode est le premier de la série à être ...,0.0000,cet épisode est le premier de la série à être ...,0.0000,cet épisode est le premier de la série à être ...,0.0000
6,7,common_voice_fr_20468203.mp3,elle est située en pleine campagne à un kilomè...,it is located in the middle of the countryside...,1.0769,elle est située en pleine campagne à un kilomè...,0.0000,elle est située en pleine campagne à un kilomè...,0.0000,elle est située en pleine campagne à un kilomè...,0.0769
7,8,common_voice_fr_19677731.mp3,le mouvement s étend à d autres ateliers dans ...,the movement extends to other workshops in the...,1.0000,le mouvement s étend à d autres ateliers dans ...,0.0000,le mouvement s étend à d autres ateliers dans ...,0.0000,le mouvement s étend à d autres ateliers dans ...,0.0000
8,9,common_voice_fr_18747257.mp3,la parole est à monsieur didier martin,the floor is for mr didier martin,0.7143,la parole est à monsieur didier martin,0.0000,la parole est à monsieur didier martin,0.0000,la parole est à monsieur didier martin,0.0000
9,10,common_voice_fr_19607910.mp3,son nom fait référence à siméon de jérusalem,his name refers to simeon of jerusalem,1.0000,son nom fait référence à siméon de jérusalem,0.0000,son nom fait référence à siméon de jérusalem,0.0000,son nom fait référence à siméon de jérusalem,0.0000
